In [ ]:
# imports
!pip install openai
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from openai import OpenAI
import matplotlib.pyplot as plt

In [ ]:
# cleaned version of data pulled from https://www.kaggle.com/datasets/stefanoleone992/rotten-tomatoes-movies-and-critic-reviews-dataset
reviews = pd.read_csv("rotten_tomatoes_reviews.csv")
reviews.head()

# Section 1: Exploratory data analysis
In the cell(s) below, answer the following questions:

    1) How many total movies are in the data set? How many "Fresh" reviews? "Rotten"?
    2) What is the average number of words in a review? Max words? Min words?
    3) Make two histograms - one for word length of "Rotten" reviews and one for word length of "Fresh" reviews. Is there a noticable difference between word length for "Rotten" and "Fresh" reviews?

# Section 2: "Old-fashioned" Natural Language Processing

Unlike all of the other data sets we have used for model building in this class, this data set has only text. *Back in my day* we didn't have fancy LLMs that could handle text with ease - we had to use other approaches to make our data model-able. Recall that our machine learning models only like to work with numbers, not raw text. We need to use a *text encoding* method to represent our raw text as a vector of numbers. This lab will walk you through two such approaches:

- bag-of-words
- n-gram bag-of-words

## Section 2a: Bag-of-words

Definitions:

- Document: a single piece of data being analyzed. In our case, a "document" is a movie review
- n: the number of documents in your data set
- Vocabulary: the set of all unique words in all documents of your data set
- m: the number of unique words across all documents in your data set (i.e., the size of your vocabulary).

The basic idea behind bag-of-words encoding is to create a *m*-length vector representing each of your documents. These *n* total vectors can be combined into a single bag-of-words matrix. For example, if your data set is two documents:

- The movie was very good.
- The movie was not bad.

Your vocabulary might be:

["the", "movie", "was", "very", "good", "not", "bad"]

The bag-of-words matrix (counts) would be:

| Document | the | movie | was | very | good | not | bad |
|-----------|-----|--------|-----|------|------|-----|-----|
| "The movie was very good." | 1 | 1 | 1 | 1 | 1 | 0 | 0 |
| "The movie was not bad."   | 1 | 1 | 1 | 0 | 0 | 1 | 1 |

**Your first task is to create this bag-of-words matrix for the movie review data set.** A couple of final things to note:

* The CountVectorizor in sklearn might prove useful here
* In theory we should do some initial cleaning on the review data to remove punctuation and make each word lowercase (therefore "the" and "THE!!" should both get counted as "the" in your vocabulary) similar to what we did with the search terms data earlier in the semester. However, CountVectorizer does this for us by default!
  * Bag-of-words representations are "sparse" (i.e., they are mostly 0s). Instead of storing all of these 0s, CountVectorizor stores the non-zero values as something called a sparse-matrix (specifically a CSR sparse matrix, in this case). You can read more about sparse-matrices here: https://en.wikipedia.org/wiki/Sparse_matrix
* Once you have created the bag-of-words matrix:
    * print out the dimensions of said matrix (you might want to check into the vocabulary_ attribute of the vectorizer to verify that the number of columns matches the vocabulary size)
    * Convert the first five rows of the CSR representation into a DataFrame and display the head of that DataFrame in your notebook. The sparsity should be apparent (you shouldn't see any non-zero values in what is displayed) and you should notice the presence of both numbers and non-english words in our 

### Once you have your vectorized/matrix representation of the reviews, we can now do some machine learning! 
In the cell(s) below, complete the following tasks:

- Divide your data into an 80/20 train/test split
    - Note: you will have to be careful about how you do this, as the "data" for your model is now in the bag-of-words output from the CountVectorizer, but the "labels" are still in your original data frame.
    - Set a random_state so that you can create the same train/test split in the next part.
- Build a Logistic Regression model using SKLearn on your training set and evaluate it on your testing set. Logistic Regression is a common "classical" approach for solving binary classification problems (despite having 'regression' in it's name, it is not a regression algorithm!). Output the accuracy of the model.
    - Note: you might need to increase the "max_iter" hyperparameter of the logistic regression model to something like 1000 (it defaults to 100).
    - **You should get an accuracy of at least 70%**

## Section 2b: n-gram bag-of-words

You should have been able to get an ok accuracy (>70%) using a simple bag-of-words approach. Can we do better? 

You may have noticed that a simple bag-of-words loses something that is very important to how the English language operates - word order. Take, for example, the following two movie reviews:

- The movie was not too bad. Pretty good actually!
- The movie was not too good. Pretty bad actually!

This results in the following bag-of-words representations: 
| Review | the | movie | was | not | too | bad | pretty | good | actually |
|:--------|:----:|:------:|:----:|:----:|:----:|:----:|:--------:|:----:|:----------:|
| **1** | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |
| **2** | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |

**They are exactly the same, despite having opposite meanings!**

One way to try to get around this problem is by introducing something called n-grams into your vocabulary, and therefore your bag-of-words representation. n-grams take each unique sequence of n words and adds that sequence to your vocabulary. Let's see the bigram (i.e., n = 2) representation of the above two movie reviews:

| Review | the movie | movie was | was not | not too | too bad | too good | pretty good | pretty bad | good actually | bad actually |
|:--------|:----------:|:----------:|:--------:|:--------:|:--------:|:--------:|:------------:|:------------:|:--------------:|:--------------:|
| **1** | 1 | 1 | 1 | 1 | 1 | 0 | 1 | 0 | 1 | 0 |
| **2** | 1 | 1 | 1 | 1 | 0 | 1 | 0 | 1 | 0 | 1 |

We can then combine the bigram vocabulary with our original vocabulary ("unigrams") and create a feature matrix that should have more context than the original bag-of-words did. Note that we could extend this out to also include trigrams, 4-grams, etc. However, the size of our feature matrix would then increase substantially (i.e., the number of unique (n+1)-word phrases in our data set is likely much larger than the number of unique n-word phrases in our data set). This has other problems (e.g., sparsity, curse of dimensionality) that will be discussed in future classes.

**In the cell(s) below:**

- Create a new bag-of-words representation of the movie reviews using unigrams and bigrams. You may want to look into the 'ngram_range' parameter of the CountVectorizer to accomplish this.
    - Output the new vocabulry size. **Answer the question - how does the bigram vocabulary size compare with your original vocabulary size?**
- Using your new bag-of-words representation, divide your model into an 80/20 train/test split.
    - Use the same random_state as above so that your results are comparable between sections
- Build a Logistic Regression model on your training set, and evaluate it on your testing set. Did your model see any improvement?

## Section 2c: tf-idf n-gram bag of words

Adding n-grams should have resulted in a small improvement in your accuracy? The question remains, can we do better still? 

Bag-of-words (as we have implemented it) still has one big problem - the weighting on the words does not make sense for the task we are trying to solve. Document vectors are just word counts of the terms in each document. Words that are very common ("a", "the", "that", etc.) will have a high count despite being unimportant to classifying whether a movie review is positive or negative. Enter **TF-IDF**.

TF-IDF stands for "Term Frequency-Inverse Document Frequency". It makes two assumptions:

1) Words that appear often in *specific* documents are important for classifying the meaning of that document.
2) Words that appear often in *all* documents are unimportant for classifying the meaning of those documents.

Using these two assumptions, TF-IDF reweights all of the words to try and determine the important words in the bag-of-words matrix. You can read more about the math here: https://en.wikipedia.org/wiki/Tf%E2%80%93idf

**In the cell(s) below:**

- Create a TF-IDF bigram bag-of-words matrix. Look into the TfidfVectorizer for how to do this.
- Using your new bag-of-words representation, divide your model into an 80/20 train/test split.
    - Use the same random_state as above so that your results are comparable between sections
- Build a Logistic Regression model on your training set, and evaluate it on your testing set. Did your model any much improvement?

# Section 3: LLMs

At the start of this lab, I said "back in my day we didn't have fancy LLMs that could handle text with ease..." Now we do! Let's throw an LLM at this problem and see how it does. 

The cell below contains a function that allows you to call a Llama model that is hosted on Rosie. Note, you will need to run this code on Rosie for it to work! Also, as we are not training a model (the LLM is already trained) you will just pass in the entire data set (note, this will take ~10 minutes to run all 10,000 reviews). **You should shoot for an accuracy greater than 90%.**

You will need to write a prompt around the movie review that you want to classify as either "Fresh" or "Rotten". Here are some hints on writing effective prompts:

- The "user" message is where you put the request you are asking of the LLM - in this case something like 'is the included movie review positive or negative?'
- The "system" prompt is optional, but is where you put "rules" you want the LLM to follow in its responses. A good first rule might be something like 'only reply with the word "Fresh" for positive movie reviews, and "Rotten" for negative movie reviews.'
- Adding additional context ('here is an example of a "Fresh" review: xxxx, 'here is an example of a "Rotten" review: xxxx') can be very useful to the model. 

In [ ]:
client = OpenAI(
   base_url = "http://dh-dgxh100-2.hpc.msoe.edu:8000/v1",
   api_key = "not_used" # this field needs to be included but is ignored
)

def make_llm_call(movie_review, client):
    response = client.chat.completions.create(
        model="meta/llama-4-scout-17b-16e-instruct",
        messages=[
                #{"role": "system", "content": "pass"}, #commented out, but you might want to include it
                {"role": "user", "content": message}
            ],
        max_tokens=2, #can make larger, but you only are likely only looking for "Fresh" or "Rotten" as response
        stream=False
    )
    return response.choices[0].message.content

# Discussion questions

- How did the LLM compare with the "classical" machine learning approaches tried in Section 2?
- Are there any drawbacks to using an LLM for this approach? Would this problem scale well if we used the entire original dataset linked at the start of the lab (that contained 1,130,017 reviews).
- Look through some of the predictions that the LLM got wrong. Include a few examples in your answer. Do you understand how the LLM made the mistake that it did?

This lab was created by Dr. Gabe Wright